In [1]:
from datetime import datetime
import os


### Choose a model_type
# model_type = 'multil'
# model_type = 'sonoisa'
# model_type = 'cl_tohoku'
model_type = 'izumi_lab'


### Set the model name
model_name = ""
if model_type == 'multil':
    model_name = 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2'
elif model_type == 'sonoisa':
    model_name = 'sonoisa/sentence-bert-base-ja-mean-tokens-v2'
elif model_type == 'cl_tohoku':
    model_name = 'cl-tohoku/bert-base-japanese-whole-word-masking'
elif model_type == 'izumi_lab':
    model_name = 'izumi-lab/bert-small-japanese'

# model_name = './tuned_models/cl_tohoku_20230403_101608'    
# model_name = './tuned_models/cl_tohoku_20230418_071425_jppost'    

print(f'{model_name = }')

    
### Set the directory path for output and create the directory
model_save_path = './tuned_models/' + model_type + '_' + datetime.now().strftime("%Y%m%d_%H%M%S")  + '_jppost'
print(model_save_path)

os.mkdir(model_save_path)

model_name = 'izumi-lab/bert-small-japanese'
./tuned_models/izumi_lab_20230420_014157_jppost


In [2]:
import json
import pandas as pd
from sentence_transformers import InputExample


def json_dataset_to_df(filepath):
    data_dicts_list = list()
    
    with open(filepath) as fp:
        for line_str in fp:
            line_dict = json.loads(line_str)
            data_dicts_list.append(line_dict)
    
    data_df = pd.DataFrame(data_dicts_list)
    
    return data_df


def create_data_training_list(data_train_df):
    data_train_inputexamples_list = list()
    
    for index, row in data_train_df.iterrows():
        sentence1_str = row['chimei1']
        sentence2_str = row['chimei2']
        label_float = float(row['similarity']) / 5
        data_train_inputexamples_list.append(InputExample(texts=[sentence1_str, sentence2_str], label=label_float))
    
    return data_train_inputexamples_list


def create_data_validation_dict(data_df):
    sentences_1_list = data_df['sentence1'].tolist()
    sentences_2_list = data_df['sentence2'].tolist()
    labels_sr = data_df['label'].astype('float') / 5
    labels_list = labels_sr.tolist()
        
    data_valid_lists_dict = {
        'sentence1': sentences_1_list,
        'sentence2': sentences_2_list,
        'label': labels_list
    }
    return data_valid_lists_dict


In [3]:
# Create datasets for training, validation and test.
from sklearn.model_selection import train_test_split

FILEPATH_INPUT_TRAIN = './jppost/sts_chimei_pairs/sts_chimei_pairs.csv'
FILEPATH_INPUT_VALID = './datasets/jsts-v1.1/train-v1.1.json'
FILEPATH_INPUT_TEST = './datasets/jsts-v1.1/valid-v1.1.json'


### Read the training dataset file.
data_train_df = pd.read_csv(FILEPATH_INPUT_TRAIN)
print(data_train_df.head(3))

# Format the data for training.
data_train_inputexamples_list = create_data_training_list(data_train_df)

# print(len(data_train_inputexamples_list))
print(data_train_inputexamples_list[:3])


### Read the validation dataset file.
data_valid_read_df = json_dataset_to_df(FILEPATH_INPUT_VALID)
print(data_valid_read_df.head(3))

# Divide data into a ratio 1 to 9.
data_valid_nouse_df, data_valid_df = train_test_split(data_valid_read_df, test_size=0.1, shuffle=False)

# Format the data for training and validation.
data_valid_lists_dict = create_data_validation_dict(data_valid_df)

print(len(data_valid_lists_dict['sentence1']))
print(data_valid_lists_dict['sentence1'][:3])
print(len(data_valid_lists_dict['sentence2']))
print(data_valid_lists_dict['sentence2'][:3])
print(len(data_valid_lists_dict['label']))
print(data_valid_lists_dict['label'][:3])


### Read the test dataset file and convert them to the dataframes.
data_test_df = json_dataset_to_df(FILEPATH_INPUT_TEST)
print(data_test_df.head(3))

# Create datasets for test.
data_test_lists_dict = create_data_validation_dict(data_test_df)

print(len(data_test_lists_dict['sentence1']))
print(data_test_lists_dict['sentence1'][:3])
print(len(data_test_lists_dict['sentence2']))
print(data_test_lists_dict['sentence2'][:3])
print(len(data_test_lists_dict['label']))
print(data_test_lists_dict['label'][:3])

   Unnamed: 0 chimei1 chimei2  similarity
0           0     北海道     北海道         5.0
1           1     山梨県     山梨県         5.0
2           2     京都府     京都府         5.0
[<sentence_transformers.readers.InputExample.InputExample object at 0x7fc279e3ce50>, <sentence_transformers.readers.InputExample.InputExample object at 0x7fc3884f2c70>, <sentence_transformers.readers.InputExample.InputExample object at 0x7fc279e3c700>]
  sentence_pair_id             yjcaptions_id               sentence1  \
0                0  10005_480798-10996-92616  川べりでサーフボードを持った人たちがいます。   
1                1      100124-104404-104405  二人の男性がジャンボジェット機を見ています。   
2                2      100142-104431-104432      男性が子供を抱き上げて立っています。   

               sentence2  label  
0  トイレの壁に黒いタオルがかけられています。    0.0  
1   2人の男性が、白い飛行機を眺めています。    3.8  
2   坊主頭の男性が子供を抱いて立っています。    4.0  
1246
['スキーをしていた人が転んでいます。', '白を基調として、タイルの色を模様にした洗面所です。', '二人の子供が掲載された紙面です。']
1246
['雪を巻き上げて、スキーヤーが疾走している。', 'タイルの色づかいでグリーンに近い台がある白い洗面所に、こげ茶色の敷物がしかれています。', 

In [ ]:
# Get a pre-trained model
from sentence_transformers import SentenceTransformer, losses, models

transformer = models.Transformer(model_name)

pooling = models.Pooling(transformer.get_word_embedding_dimension(), pooling_mode='mean')
model = SentenceTransformer(modules=[transformer, pooling], device='cuda')

# Define your train dataset, the dataloader and the train loss
train_loss = losses.CosineSimilarityLoss(model)

In [ ]:
from sentence_transformers import evaluation
from torch.utils.data import DataLoader


dataloader_train = DataLoader(
    dataset = data_train_inputexamples_list,
    batch_size = 512,
    # shuffle = False,
    shuffle = True,
    sampler = None,
    batch_sampler = None,
    num_workers = 0,
    collate_fn = None,
    pin_memory = False,
    drop_last = False,
    timeout = 0,
    worker_init_fn = None,
    prefetch_factor = 2,
    persistent_workers = False
)
        
evaluator = evaluation.EmbeddingSimilarityEvaluator(
    sentences1 = data_valid_lists_dict['sentence1'],
    sentences2 = data_valid_lists_dict['sentence2'],
    scores = data_valid_lists_dict['label'],
    batch_size = 32,
    main_similarity = evaluation.SimilarityFunction.COSINE,
    name = '',
    show_progress_bar = True,
    write_csv = True
)
    
model.fit(
    train_objectives = [(dataloader_train, train_loss)],
    evaluator = evaluator,
    epochs = 3,
    steps_per_epoch = None,
    scheduler = 'WarmupLinear',
    # warmup_steps = 10000,  # training data/100
    warmup_steps = 50,
    # optimizer_class = <class 'torch.optim.adamw.AdamW'>,
    optimizer_params = {'lr': 2e-05},
    weight_decay = 0.01,
    evaluation_steps = 0,
    output_path = model_save_path,
    save_best_model = True,
    max_grad_norm = 1,
    use_amp = False,
    callback = None,
    show_progress_bar = True,
    checkpoint_path = None,
    checkpoint_save_steps = 500,
    checkpoint_save_total_limit = 0
)

In [6]:
from sklearn.metrics.pairwise import paired_cosine_distances

# Encode(Embed) sentence data
embeddings1_list = model.encode(data_test_lists_dict['sentence1'])
embeddings2_list = model.encode(data_test_lists_dict['sentence2'])

cossims_list = 1 - (paired_cosine_distances(embeddings1_list, embeddings2_list))

print(cossims_list)

[0.7656187  0.82223606 0.9151375  ... 0.8067348  0.89091045 0.99080265]


In [7]:
from scipy.stats import pearsonr, spearmanr

score_pearson_float, _ = pearsonr(data_test_lists_dict['label'], cossims_list)
score_spearman_float, _ = spearmanr(data_test_lists_dict['label'], cossims_list)

print(f'{score_pearson_float = }')
print(f'{score_spearman_float = }')

score_pearson_float = 0.6409547291552374
score_spearman_float = 0.6388223891492841


In [8]:
### izumi_lab_20230420_014157_jppost
# score_pearson_float = 0.6409547291552374
# score_spearman_float = 0.6388223891492841

### cl_tohoku_20230418_071425_jppost
# score_pearson_float = 0.8540016341478083
# score_spearman_float = 0.8084751514584866

### izumi_lab_20230418_095436_jppost
# score_pearson_float = 0.8065565240910519
# score_spearman_float = 0.7547151747186321


### multilingual
## Before training
# score_pearson_float = 0.8395266716973665
# score_spearman_float = 0.7946157299720272
# 
## epoch:3, batch-size: 1; 20 mins/epoch, multil_20230404_021123
# score_pearson_float = 0.8591599779142202
# score_spearman_float = 0.8192690043101212


### sonoisa
## Before training
# score_pearson_float = 0.8611210755385836
# score_spearman_float = 0.8086585777543251
# 
## epoch:3, batch-size: 1; 14 mins, sonoisa_20230403_044056
# score_pearson_float = 0.8787569421662808
# score_spearman_float = 0.8358817280739163


### cl-tohoku
## Before training
# score_pearson_float = 0.6868976163285792
# score_spearman_float = 0.6736940402774908
# 
## epoch:3, batch-size: 1; 14 mins, cl_tohoku_20230403_065812
# score_pearson_float = 0.877305315180833
# score_spearman_float = 0.8327534615755807
# 
# 
## epoch:3, batch-size: 16; 1 min, cl_tohoku_20230403_101608
# score_pearson_float = 0.8776290830127422
# score_spearman_float = 0.832529946778794
# 
## 5-Fold, epoch:1, batch-size: 16, cl_tohoku_20230401_040019
# score_pearson_float = 0.8797983013477866
# score_spearman_float = 0.8350105828169487
# 

### izumi-lab
## Before training
# score_pearson_float = 0.6404931475426449
# score_spearman_float = 0.6329425227142002
# 
## epoch:1
# score_pearson_float = 0.8112839175373776
# score_spearman_float = 0.7570156536248307
# 
## epoch:3, batch-size: 1; 14 mins, izumi_lab_20230403_055128
# score_pearson_float = 0.8517744341835913
# score_spearman_float = 0.8023957303081923
# 
## epoch:3, batch-size: 16; 1 min, izumi_lab_20230403_095712
# score_pearson_float = 0.83373046934419
# score_spearman_float = 0.7783581857305379
# 
## 5-Fold, epoch:1, batch-size: 16, izumi_lab_20230401_034759
# score_pearson_float = 0.8421832121810837
# score_spearman_float = 0.7870142697712181
# 